In [3]:
from statistics import mean

import pandas as pd

print(pd.__version__)
df=pd.read_csv("amazon_ecommerce_1M.csv")

3.1.1


In [7]:
df['category']=df[('category')].astype('category')
df['purchase_date']=pd.to_datetime(df['purchase_date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 20 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   user_id             1000000 non-null  str           
 1   product_id          1000000 non-null  str           
 2   category            1000000 non-null  category      
 3   subcategory         1000000 non-null  str           
 4   brand               1000000 non-null  str           
 5   price               1000000 non-null  float64       
 6   discount            1000000 non-null  float64       
 7   final_price         1000000 non-null  float64       
 8   rating              1000000 non-null  float64       
 9   review_count        1000000 non-null  int64         
 10  stock               1000000 non-null  int64         
 11  seller_id           1000000 non-null  str           
 12  seller_rating       1000000 non-null  float64       
 13  purchase_date       1000

In [11]:
df.sample(5)

,user_id,product_id,category,subcategory,brand,price,discount,final_price,rating,review_count,stock,seller_id,seller_rating,purchase_date,shipping_time_days,location,device,payment_method,is_returned,delivery_status
972178,U951745,P85342,Home,Furniture,Boat,2644.15,41.49,1547.15,3.8,36,445,S7332,2.9,2026-03-01,4,Chennai,Web,Credit Card,False,In Transit
221557,U861415,P33760,Home,Decor,Zara,12372.36,27.06,9024.62,3.7,60,100,S2004,2.8,2024-06-24,3,Chennai,Web,Cash on Delivery,True,Returned
999956,U861353,P84762,Home,Decor,LG,9683.79,14.78,8252.98,2.8,7,147,S7959,3.5,2025-05-04,4,Hyderabad,Web,Debit Card,True,Returned
18716,U232636,P99258,Beauty,Haircare,Zara,283.13,35.39,182.92,3.5,173,230,S5138,3.6,2024-08-05,5,Hyderabad,Mobile App,Debit Card,False,In Transit
349355,U644677,P77216,Clothing,Women,Zara,4376.30,10.28,3926.57,4.3,9,154,S4522,2.6,2024-08-29,1,Delhi,Tablet,Cash on Delivery,False,Delivered


In [18]:
revenue_by_category=df.groupby('category')['final_price'].sum().sort_values(ascending=False)
print('Выручка по категориям:')
revenue_by_category

category
Electronics    6.579048e+09
Home           1.529978e+09
Sports         1.124849e+09
Beauty         3.759517e+08
Clothing       3.290504e+08
Name: final_price, dtype: float64

In [34]:
return_rate_sub=df.groupby('subcategory')['is_returned'].mean()*100
print('Анализ возвратов, %')
return_rate_sub

subcategory
Camera        11.510719
Cycling       11.715475
Decor         11.769254
Fitness       11.593291
Furniture     11.566453
Haircare      11.630468
Headphones    11.077585
Kids          11.710892
Kitchen       11.415825
Laptop        11.233313
Makeup        11.842125
Men           11.824502
Mobile        11.271940
Outdoor       11.629613
Skincare      11.795310
Women         11.673387
Name: is_returned, dtype: float64

In [53]:
df['discount_size'] = pd.qcut(df['discount'], q=3, labels=['Низкая', 'Средняя', 'Высокая'])
discount_impact_new = df.groupby('discount_size').agg(
    avg_rating=('rating', 'mean'),
    total_orders=('product_id', 'count')
)
print('Влияние скидок на покупки и рейтинг')
discount_impact_new

               avg_rating  total_orders
discount_size                          
Низкая           3.953036        333533
Средняя          3.948972        333317
Высокая          3.879630        333150


In [38]:
print('Распределение статусов доставки: ')
df['delivery_status'].value_counts()

Распределение статусов доставки: 


delivery_status
Delivered     295234
Delayed       294983
In Transit    293793
Returned      115990
Name: count, dtype: int64

In [51]:
city_shipping=df.groupby('location').agg(avg_shipping_days=('shipping_time_days','mean'),total_orders=('product_id','count')).sort_values(by='avg_shipping_days')
print('Топ-2 города с самой быстрой доставкой')
display(city_shipping.head(2))
print('Топ-3 города с самой долгой доставкой')
display(city_shipping.tail(3))


Топ-2 города с самой быстрой доставкой


,avg_shipping_days,total_orders
location,,
Delhi,2.662017,200649
Mumbai,2.669277,199874


Топ-3 города с самой долгой доставкой


,avg_shipping_days,total_orders
location,,
Hyderabad,3.495336,199305
Bangalore,3.500974,200270
Chennai,3.506523,199902


In [52]:
shipping_vs_returns=df.groupby('is_returned')['shipping_time_days'].mean()
print('Проверка гипотезы связано ли желание клиента вернуть товар с тем,что доставка заняла слишком много времени')
shipping_vs_returns

is_returned
False    3.134466
True     3.409690
Name: shipping_time_days, dtype: float64